# Text Summarization Pipeline Using LangChain - Sprint 2

Este notebook implementa um pipeline de resumo de texto usando LangChain, conforme o desafio proposto.

Você encontrará funções para:
- Carregamento de texto (arquivo .txt, .pdf ou entrada manual)
- Divisão inteligente de texto (chunking com sobreposição para preservar contexto)
- Resumo com múltiplos modos (curto, detalhado, tópicos)
- Consolidação de resumos parciais (map-reduce pattern)
- Integração com LLM Groq (modelo de IA generativa)


# Seção 1: Instalação de Dependências

Foi utilizando apenas as bibliotecas necessárias para o pipeline LangChain.

Objetivo da Seção: Garantir que todas as bibliotecas LangChain estejam na versão correta e compatível com o pipeline de resumo.

In [1]:
# ================================
# INSTALAÇÃO (executar 1x)
# ================================
# Remove versões antigas que podem causar conflitos
!pip uninstall -y langchain langchain-core langchain-community langsmith langchain-groq

# Instala versões específicas e compatíveis
!pip install langchain==0.2.16 langchain-core==0.2.43 langchain-community==0.2.16 langchain-groq
!pip install langchain==0.2.16 langchain-core==0.2.43 langchain-community==0.2.16 langchain-groq==0.1.5


Found existing installation: langchain 0.2.16
Uninstalling langchain-0.2.16:
  Successfully uninstalled langchain-0.2.16
Found existing installation: langchain-core 0.2.43
Uninstalling langchain-core-0.2.43:
  Successfully uninstalled langchain-core-0.2.43
Found existing installation: langchain-community 0.2.16
Uninstalling langchain-community-0.2.16:
  Successfully uninstalled langchain-community-0.2.16
Found existing installation: langsmith 0.1.147
Uninstalling langsmith-0.1.147:
  Successfully uninstalled langsmith-0.1.147
Found existing installation: langchain-groq 0.1.5
Uninstalling langchain-groq-0.1.5:
  Successfully uninstalled langchain-groq-0.1.5
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Using cached langchain-0.2.16-py3-none-any.whl (1.0 MB)
  Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
  Using cached langchain_community-0.2.16-py3-none-any.whl (2.3 MB)
  Using cached langchain_gro

## Seção 2: Importações LangChain

Importamos os componentes essenciais para construir o pipeline.

Objetivo da Seção: Importar todos os componentes LangChain necessários para o pipeline funcionar corretamente.

In [2]:
# ================================
# IMPORTS LANGCHAIN
# ================================

# Importa template de prompts para criar instruções dinâmicas
from langchain_core.prompts import ChatPromptTemplate

# Importa o modelo de linguagem Groq (LLM)
from langchain_groq import ChatGroq

# Importa parser para converter saída do LLM em string
from langchain_core.output_parsers import StrOutputParser

# Importa divisor de texto inteligente (chunking)
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Importa carregador de PDFs
from langchain_community.document_loaders import PyPDFLoader

# Importa módulo para variáveis de ambiente
import os

## Seção 3: Configuração da API Key

Carregamos a chave da API de forma segura, sem expô-la no código.

Objetivo da Seção: Carregar a chave da API Groq de forma segura, sem expô-la no código-fonte.

In [3]:
# =========================================================
# CONFIGURAÇÃO DA API KEY (boa prática)
# =========================================================

def carregar_chave(caminho_arquivo):
    """
    Lê a chave da API a partir de um arquivo texto externo.
    
    Esta função implementa boa prática de segurança:
    - A chave não fica exposta no código
    - Pode ser facilmente alterada sem modificar o script
    - Evita commits acidentais de credenciais
    
    Parâmetros:
        caminho_arquivo (str): Caminho do arquivo contendo a chave Groq
                              Exemplo: "api_key_groq.txt"
    
    Retorna:
        str: A chave da API sem espaços em branco (strip)
    
    Exceção:
        Exception: Levanta erro se o arquivo não for encontrado
    
    Exemplo de uso:
        >>> api_key = carregar_chave("api_key_groq.txt")
        >>> print(api_key)  # Exibe a chave carregada
    """
    try:
        # Abre arquivo em modo leitura
        with open(caminho_arquivo, "r") as f:
            # Lê conteúdo e remove espaços/quebras de linha
            return f.read().strip()
    except FileNotFoundError:
        # Se arquivo não existir, levanta exceção informativa
        raise Exception("Arquivo de chave não encontrado")

# Lê a chave do arquivo externo
api_key = carregar_chave("api_key_groq.txt")

# Define a chave como variável de ambiente
# O LangChain automaticamente busca GROQ_API_KEY do ambiente
os.environ["GROQ_API_KEY"] = api_key

## Seção 4: Configuração do Modelo LLM

Inicializamos o modelo de linguagem com parâmetros otimizados para resumo.

Objetivo da Seção: Inicializar o modelo de linguagem com configurações otimizadas para resumo (temperatura baixa garante respostas consistentes e focadas).

In [4]:
# =========================================================
# CONFIGURAÇÃO DO MODELO (LLM)
# =========================================================

# Inicializa o modelo de linguagem Groq
llm = ChatGroq(
    # Modelo escolhido: Llama 3.3 com 70 bilhões de parâmetros
    # Suporta contexto amplo (até 8k tokens)
    # Versátil para múltiplas tarefas
    model="llama-3.3-70b-versatile", # atualizado
    # model="llama-3.1-8b-instant",  # atualizado e mais econômico 
    
    # Temperature controla a criatividade da resposta
    # 0.3 = baixa criatividade, respostas mais determinísticas
    # Ideal para resumo (queremos consistência, não criatividade)
    # Escala: 0.0 (determinístico) até 1.0 (criativo)
    temperature=0.3
)

## Seção 5: Modos de Resumo (Requisito do Desafio)

Criamos três modos diferentes de resumo para atender a diferentes necessidades.

Objetivo da Seção: Oferecer três modos de resumo diferentes (curto, detalhado, tópicos) para atender a diferentes necessidades de usuários, conforme requisito 3.3 do desafio.

In [5]:
# =========================================================
# MODOS DE RESUMO (REQUISITO DO DESAFIO)
# =========================================================

def criar_prompt(modo="curto"):
    """
    Cria um prompt dinâmico baseado no modo de resumo selecionado.
    
    Esta função implementa o requisito 3.3 do desafio:
    "O engine deve produzir pelo menos um dos seguintes:
     - Short Summary
     - Detailed Summary"
    
    Adicionamos um terceiro modo (tópicos) para maior flexibilidade.
    
    Parâmetros:
        modo (str): Tipo de resumo desejado
                   - "curto": TL;DR (muito breve, 2-3 linhas)
                   - "detalhado": Resumo completo com contexto
                   - "topicos": Bullet points dos pontos principais
    
    Retorna:
        ChatPromptTemplate: Template de prompt configurado para o modo
                           Pronto para ser usado em uma chain LangChain
    
    Exceção:
        Exception: Se o modo for inválido (não está na lista acima)
    
    Exemplo de uso:
        >>> prompt = criar_prompt("curto")
        >>> chain = prompt | llm | parser
        >>> resultado = chain.invoke({"input": "texto longo..."})
    """
  # =====================================================
    # MODO 1: RESUMO CURTO (TL;DR)
    # =====================================================
    if modo == "curto":
        # Template para resumo muito breve
        # Ideal para: executivos com pouco tempo, notificações rápidas
        template = """
        Resuma o texto de forma curta e objetiva (TL;DR).
        Foque apenas nos pontos principais.
        Máximo 3-4 linhas.
      Texto:
        {input}
        """
  # =====================================================
    # MODO 2: RESUMO DETALHADO
    # =====================================================
    elif modo == "detalhado":
        # Template para resumo completo com contexto
        # Ideal para: análise profunda, documentação, relatórios
        template = """
        Faça um resumo detalhado do texto abaixo, incluindo:
        - contexto geral
        - pontos principais
        - decisões tomadas
        - ações recomendadas
      Texto:
        {input}
        """
  # =====================================================
    # MODO 3: RESUMO EM TÓPICOS (BULLET POINTS)
    # =====================================================
    elif modo == "topicos":
        # Template para extração de pontos-chave em formato estruturado
        # Ideal para: apresentações, reuniões, quick reference
        template = """
        Extraia os principais pontos do texto em formato de bullet points:
      - pontos principais
        - decisões tomadas
        - ações necessárias
        - próximos passos
      Texto:
        {input}
        """
  # =====================================================
  # MODO INVÁLIDO
  # =====================================================
    else:
        # Se modo não for reconhecido, levanta erro
        raise Exception("Modo inválido. Use: 'curto', 'detalhado' ou 'topicos'")
  # Converte string template em ChatPromptTemplate do LangChain
    # {input} será substituído pelo texto real quando invocar a chain
    return ChatPromptTemplate.from_template(template)  

## Seção 6: Leitura de Texto (Arquivo + Fallback Manual)

Implementamos suporte para múltiplas fontes de entrada com tratamento robusto de erros.

Objetivo da Seção: Implementar o requisito 3.1 (Input Requirements) — aceitar texto de múltiplas fontes (.txt, .pdf, entrada manual) com tratamento robusto de erros e fallback inteligente.

In [6]:
# =========================================================
# LEITURA DE TEXTO (arquivo + fallback manual)
# =========================================================

def ler_texto(caminho_arquivo=None, texto_manual=None):
    """
    Lê texto de múltiplas fontes com tratamento inteligente de erros.
    
    Esta função implementa o requisito 3.1 do desafio:
    "O tool deve aceitar arbitrary-length text.
     Input pode ser: pasted text, arquivo .txt, PDF via LangChain loader"
    
    Estratégia de fallback:
    1. Tenta carregar arquivo se caminho fornecido
    2. Se arquivo vazio, usa texto manual como fallback
    3. Se arquivo não existe, usa texto manual
    4. Se nenhum disponível, levanta erro
    
    Parâmetros:
        caminho_arquivo (str, opcional): Caminho para arquivo .txt ou .pdf
                                        Exemplo: "reuniao.pdf" ou "relatorio.txt"
        texto_manual (str, opcional): Texto fornecido diretamente no código
                                     Usado como fallback se arquivo falhar
    
    Retorna:
        str: Conteúdo do texto processado e limpo
    
    Exceção:
        Exception: Se nenhuma fonte válida for fornecida
    
    Suporta:
        - Arquivos .txt (leitura direta com encoding UTF-8)
        - Arquivos .pdf (via LangChain PyPDFLoader)
        - Texto manual como fallback
    
    Exemplo de uso:
        >>> texto = ler_texto(
        ...     caminho_arquivo="reuniao.pdf",
        ...     texto_manual="Texto de fallback..."
        ... )
    """
  # =====================================================
    # ETAPA 1: TENTAR CARREGAR ARQUIVO
    # =====================================================
    if caminho_arquivo:
        try:
            # Extrai extensão do arquivo (ex: ".txt", ".pdf")
            extensao = os.path.splitext(caminho_arquivo)[1].lower()
          # =====================================================
            # CASO 1: ARQUIVO .TXT
            # =====================================================
            if extensao == ".txt":
                # Abre arquivo em modo leitura com encoding UTF-8
                # UTF-8 suporta caracteres especiais (acentos, símbolos)
                with open(caminho_arquivo, "r", encoding="utf-8") as f:
                    # Lê conteúdo e remove espaços extras nas bordas
                    conteudo = f.read().strip()
          # =====================================================
            # CASO 2: ARQUIVO .PDF
            # =====================================================
            elif extensao == ".pdf":
                # Usa LangChain PyPDFLoader para extrair texto de PDF
                # Suporta múltiplas páginas automaticamente
                loader = PyPDFLoader(caminho_arquivo)
                
                # Load retorna lista de Document objects
                # Cada documento representa uma página
                documentos = loader.load()
              # Junta conteúdo de todas as páginas com quebra de linha
                # Preserva estrutura do documento original
                conteudo = "\n".join([doc.page_content for doc in documentos]).strip()
          # =====================================================
            # CASO 3: EXTENSÃO NÃO SUPORTADA
            # =====================================================
            else:
                # Se extensão não é .txt nem .pdf, levanta erro
                raise Exception("Formato não suportado. Use .txt ou .pdf")
          # =====================================================
            # VALIDAÇÃO: ARQUIVO TEM CONTEÚDO?
            # =====================================================
            if conteudo:
                # Arquivo carregado com sucesso
                print(f"Usando arquivo ({extensao})")
                return conteudo
          # =====================================================
            # FALLBACK 1: ARQUIVO VAZIO, USA TEXTO MANUAL
            # =====================================================
            elif texto_manual:
                print("Arquivo vazio, usando texto manual como fallback")
                return texto_manual.strip()
          # =====================================================
            # ERRO: ARQUIVO VAZIO E SEM FALLBACK
            # =====================================================
            else:
                raise Exception("Arquivo vazio e sem texto manual fornecido")
      # =====================================================
        # TRATAMENTO DE ERRO: ARQUIVO NÃO ENCONTRADO
        # =====================================================
        except FileNotFoundError:
            # Se arquivo não existe, tenta usar texto manual
            if texto_manual:
                print("Arquivo não encontrado, usando texto manual como fallback")
                return texto_manual.strip()
            else:
                # Se não há fallback, levanta erro informativo
                raise Exception(f"Arquivo '{caminho_arquivo}' não encontrado")
  # =====================================================
    # ETAPA 2: USAR TEXTO MANUAL (FALLBACK)
    # =====================================================
    if texto_manual:
        print("Usando texto manual fornecido")
        return texto_manual.strip()
  # =====================================================
    # ERRO: NENHUMA FONTE DISPONÍVEL
    # =====================================================
    raise Exception("Informe um texto manual ou um caminho de arquivo válido")

## Seção 7: Chunking (Divisão Inteligente de Texto)

Dividimos textos longos em partes gerenciáveis, preservando contexto.

Objetivo da Seção: Implementar o requisito 3.2 (Chunking & Pre-processing) — dividir textos longos em partes gerenciáveis, preservando contexto através da sobreposição.

In [13]:
# =========================================================
# CHUNKING (REQUISITO OFICIAL)
# =========================================================

def dividir_texto(texto):
    """
    Divide texto em chunks gerenciáveis usando RecursiveCharacterTextSplitter.
    
    Esta função implementa o requisito 3.2 do desafio:
    "Use a LangChain text splitter (ex: RecursiveCharacterTextSplitter).
     Chunk length, overlap, e processing strategy devem ser definidos e justificados.
     O sistema deve evitar perder contexto entre chunks."
    
    ESTRATÉGIA DE CHUNKING:
    
    1. TAMANHO DO CHUNK (chunk_size=2000):
       - 2000 caracteres ≈ 500 tokens (aproximadamente)
       - Modelo Groq suporta até 8000 tokens de contexto
       - 2000 chars deixa espaço para prompt + resposta
       - Equilibra: contexto suficiente vs. limite de tokens
    
    2. SOBREPOSIÇÃO (chunk_overlap=200):
       - 200 caracteres de sobreposição entre chunks consecutivos
       - Evita perda de contexto nas bordas dos chunks
       - Exemplo: Chunk 1 termina com "...decisão foi..."
                  Chunk 2 começa com "...foi tomada porque..."
       - Garante continuidade lógica
    
    3. RECURSIVE CHARACTER SPLITTER:
       - Respeita limites naturais do texto (parágrafos, frases)
       - Não quebra palavras no meio
       - Mantém estrutura semântica do documento
       - Melhor que split simples por caracteres
    
    Parâmetros:
        texto (str): Texto longo a ser dividido
                    Pode ter qualquer tamanho
    
    Retorna:
        list: Lista de strings (chunks) prontos para resumo
              Exemplo: ["Chunk 1 com 2000 chars...", "Chunk 2 com 2000 chars...", ...]
    
    Exemplo de uso:
        >>> texto_longo = "Este é um texto muito longo com..."
        >>> chunks = dividir_texto(texto_longo)
        >>> print(f"Dividido em {len(chunks)} partes")
        Dividido em 5 partes
    """
  # Inicializa o splitter com configurações otimizadas
    splitter = RecursiveCharacterTextSplitter(
        # Tamanho máximo de cada chunk em caracteres
        # Justificativa: 2000 chars ≈ 500 tokens, deixa margem para LLM
        chunk_size=12000,
        
        # Número de caracteres que se sobrepõem entre chunks
        # Justificativa: 200 chars preserva contexto nas bordas
        chunk_overlap=1200
    )
  # Executa a divisão e retorna lista de chunks
    # Cada chunk é uma string independente
    return splitter.split_text(texto)

## Seção 8: Pipeline de Resumo (Map-Reduce)

Implementamos o padrão Map-Reduce para resumir textos longos.

Objetivo da Seção: Implementar o padrão Map-Reduce (requisito 3.3) — resumir cada chunk individualmente e depois consolidar em um resumo final único.

In [14]:
# =========================================================
# PIPELINE DE RESUMO
# =========================================================

def resumir_texto(texto, modo="curto"):
    """
    Pipeline completo de resumo usando padrão Map-Reduce.
    
    Esta função implementa o requisito 3.3 do desafio:
    "Use LangChain summarization chains como load_summarize_chain
     (map-reduce recomendado) ou custom LLMChain strategy"
    
    FLUXO DO PIPELINE (Map-Reduce):
    
    1. CHUNKING:
       - Divide texto longo em partes gerenciáveis
       - Preserva contexto com sobreposição
       - Exemplo: 10 páginas → 5 chunks de 2000 chars
    
    2. MAP STEP (Resumir cada parte):
       - Processa cada chunk individualmente
       - Aplica o mesmo prompt a todos os chunks
       - Gera resumo parcial para cada parte
       - Exemplo: 5 chunks → 5 resumos parciais
    
    3. REDUCE STEP (Consolidar):
       - Junta todos os resumos parciais
       - Aplica o prompt novamente ao texto consolidado
       - Gera resumo final único
       - Exemplo: 5 resumos parciais → 1 resumo final
    
    Vantagens do Map-Reduce:
    - Suporta textos maiores que o contexto do LLM
    - Cada chunk é processado independentemente
    - Consolidação garante coerência final
    - Escalável para documentos muito longos
    
    Parâmetros:
        texto (str): Texto longo a ser resumido
                    Pode ter qualquer tamanho
        modo (str): Modo de resumo desejado
                   - "curto": TL;DR breve
                   - "detalhado": Resumo completo
                   - "topicos": Bullet points
    
    Retorna:
        str: Resumo final consolidado
             Texto único e coerente
    
    Exemplo de uso:
        >>> texto_longo = "Este é um documento de 50 páginas..."
        >>> resumo = resumir_texto(texto_longo, modo="detalhado")
        >>> print(resumo)
        "Resumo consolidado do documento..."
    """
  # =====================================================
    # ETAPA 1: PREPARAR CHAIN (Prompt + LLM + Parser)
    # =====================================================
    
    # Cria prompt dinâmico baseado no modo selecionado
    prompt = criar_prompt(modo)
    
    # Parser converte saída do LLM em string simples
    parser = StrOutputParser()
    
    # Chain = Prompt → LLM → Parser
    # Usa pipe operator (|) do LangChain para composição
    # Quando invocar: texto entra no prompt, LLM processa, parser extrai string
    chain = prompt | llm | parser
  # =====================================================
    # ETAPA 2: CHUNKING (Dividir texto)
    # =====================================================
    
    # Divide texto em partes gerenciáveis
    partes = dividir_texto(texto)
    
    # Exibe informação sobre divisão
    print(f"\n🔹 Texto dividido em {len(partes)} partes\n")
  # Lista para armazenar resumos parciais
    resumos_parciais = []
  # =====================================================
    # ETAPA 3: MAP STEP (Resumir cada chunk)
    # =====================================================
    
    # Itera sobre cada chunk
    for i, parte in enumerate(partes):
        # Exibe progresso do processamento
        print(f"Processando parte {i+1}/{len(partes)}...")
        
        # Invoca chain com o chunk atual
        # Prompt substitui {input} pelo conteúdo do chunk
        # LLM gera resumo do chunk
        # Parser extrai string do resultado
        resumo_parte = chain.invoke({"input": parte})
        
        # Armazena resumo parcial
        resumos_parciais.append(resumo_parte)
  # =====================================================
    # ETAPA 4: REDUCE STEP (Consolidar resumos)
    # =====================================================
    
    # Junta todos os resumos parciais em um único texto
    # Usa quebra de linha para separar resumos
    texto_consolidado = "\n".join(resumos_parciais)
  # Exibe informação sobre consolidação
    print("\n Consolidando resumo final...\n")
  # Invoca chain novamente com texto consolidado
    # Prompt substitui {input} pelo texto consolidado
    # LLM gera resumo final do texto consolidado
    # Parser extrai string do resultado
    resumo_final = chain.invoke({"input": texto_consolidado})
  # Retorna resumo final único e coerente
    return resumo_final

## Seção 9: Execução (Exemplo Prático)

Demonstramos o uso prático do pipeline com um exemplo real.

Objetivo da Seção: Demonstrar o uso prático do pipeline com um exemplo real (reunião), atendendo ao requisito 5.C (Example Input and Output).

In [15]:
# =========================================================
# EXECUÇÃO (EXEMPLO)
# =========================================================

# Carrega texto de múltiplas fontes
# Tenta carregar arquivo PDF, usa texto manual como fallback
texto = ler_texto(
    # Tenta carregar arquivo PDF / txt
    caminho_arquivo="reuniao.txt",
    
    # Se o arquivo não existir ou estiver vazio, usa este texto
    texto_manual="""
    Na reunião foi discutido o atraso na entrega do projeto X.
    João ficou responsável por ajustar o cronograma.
    Maria irá revisar os dados até sexta-feira.
    Também foi comentado sobre a melhoria no processo interno.
    """
)

# Seleciona modo de resumo
# Opções: "curto" | "detalhado" | "topicos"
modo_resumo = "topicos"

# Executa pipeline de resumo
# Retorna resumo final consolidado
resultado = resumir_texto(texto, modo=modo_resumo)

# Exibe resultado formatado
print("\n===== RESUMO FINAL =====\n")
print(resultado)

Usando arquivo (.txt)

🔹 Texto dividido em 5 partes

Processando parte 1/5...
Processando parte 2/5...
Processando parte 3/5...
Processando parte 4/5...
Processando parte 5/5...

 Consolidando resumo final...


===== RESUMO FINAL =====

Aqui estão os principais pontos do texto em formato de bullet points:

**Pontos Principais:**
* Discussão sobre a criação de um painel de controle para acompanhamento de compromissos
* Utilização de ícones para representar diferentes status de compromisso
* Definição de regras para a utilização de ícones e cores
* Discussão sobre a criação de um gráfico e a forma como os dados devem ser apresentados
* Análise de diferentes opções de design e layout

**Decisões Tomadas:**
* Utilizar ícones amarelos para representar pontos de atenção
* Manter apenas a linha do diagnóstico e remover a linha da proposta
* Criar uma nova aba para apresentar as proposições de revisão
* Manter o mesmo padrão de design para todos os visuais
* Ajustar o tamanho e a cor de elemen